In [1]:
from Environment import *
from DDPG import *
from NN_Module import *
import os

import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from numpy import linalg as LA

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
from pandapower.plotting.plotly import pf_res_plotly

from loguru import logger

### a simple logger
logger.remove()
logger.add(sys.stderr, level='INFO')

4

In [2]:
env_seed = 7        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l
episode_num = 200   # the total test episode
step_num = 200      # the longest test step

### create testing environment
injection_bus = np.array([18, 21, 30, 45, 53])-1
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)
state, topology, senario = env.reset_topo(seed=env_seed)
topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
# pf_res_plotly(pp_net);

### Some Plot Function

In [7]:
def moving_average(a, n=3):
    # Padding the array to maintain the length after convolution.
    pad = np.pad(a, (n//2, n-1-n//2), mode='edge')
    ret = np.convolve(pad, np.ones(n), mode='valid') / n
    return ret

# plot policy
def policy_plotly(policy_net, topology):
    """
    用 Plotly 绘制各母线的策略曲线，每个子图显示一个母线的 RLC-FT 策略与基线（Linear）策略比较，
    """
    default_colors = pc.qualitative.Plotly  # Plotly 默认颜色序列
    color_linear = default_colors[0]
    color_rlc = default_colors[1]
    fig = make_subplots(rows=1, cols=5,
                        subplot_titles=['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53'])
    N = 400
    for i in range(5):
        baseline_vals = []
        policy_vals = []
        for j in range(N):
            # 计算基线控制值：baseline = max(state-1.05, 0) - max(0.95-state, 0)
            state_val = 0.80 + 0.001 * j
            base = np.maximum(state_val - 1.05, 0) - np.maximum(0.95 - state_val, 0)
            baseline_vals.append(-base)  # 取负值
            state_tensor = torch.tensor([[state_val]])
            action_tensor = policy_net[i](state_tensor, topology)
            policy_vals.append(float(-action_tensor.detach().cpu().numpy()[0]))

        baseline_vals = np.array(baseline_vals)
        policy_vals_smoothed = moving_average(np.array(policy_vals), n=20)
        baseline_vals_scaled = 5 * baseline_vals
        
        x_vals = np.linspace(10, 14, N)
        
        # 仅在第一列显示图例，其余子图同组 trace 设为不显示图例
        showlegend = True if i == 0 else False

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=baseline_vals_scaled,
            mode='lines',
            name='Linear',
            legendgroup='Linear',
            showlegend=showlegend,
            line=dict(dash='dash', color=color_linear)
        ), row=1, col=i+1)

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=policy_vals_smoothed,
            mode='lines',
            name='RLC-FT',
            legendgroup='RLC-FT',
            showlegend=showlegend,
            line=dict(color=color_rlc)
        ), row=1, col=i+1)

    # 保证仅在第一个子图显示y轴标题，第三个子图显示x轴标题
    fig.update_yaxes(title_text="Q (MVar)", row=1, col=1)
    fig.update_xaxes(title=dict(text="Voltage (kV)", standoff=25), row=1, col=3)
    fig.update_layout(
        width=1400,
        height=500,
        showlegend=True,
        font=dict(size=16),
        xaxis=dict(
            tickfont=dict(size=12),
            showline=True,
            mirror=True,
            showgrid=True,
        ),
        yaxis=dict(
            tickfont=dict(size=12),
            showline=True,
            mirror=True,
            showgrid=True,
        ),
    )
    
    output_path = os.path.join(Config.data_path, 'images', '56bus', 'policy_plot.pdf')
    import plotly.io as pio
    pio.kaleido.scope.mathjax = None
    fig.write_image(output_path)
    fig.show()


def safe_net_plotly(safe_net):
    """
    用 Plotly 绘制 safe network 策略曲线，每个子图显示一个母线的 Stable-DDPG 与 Linear 比较
    """
    default_colors = pc.qualitative.Plotly  # Plotly 默认颜色序列
    color_linear = default_colors[0]
    color_safe = default_colors[1]
    fig = make_subplots(rows=1, cols=5,
                        subplot_titles=['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53'])
    N = 400
    for i in range(len(safe_net)):
        baseline_vals = []
        safe_vals = []
        for j in range(N):
            state_val = 0.80 + 0.001 * j
            base = np.maximum(state_val - 1.05, 0) - np.maximum(0.95 - state_val, 0)
            baseline_vals.append(-base)
            # safe_net[i].get_action 接受列表输入，返回单个数值
            action = safe_net[i].get_action([state_val])
            safe_vals.append(-float(action))
        baseline_vals = np.array(baseline_vals)
        baseline_vals_scaled = 2 * baseline_vals
        x_vals = np.linspace(10, 14, N)
        # 仅在第一列显示图例，其余子图同组 trace 设为不显示图例
        showlegend = True if i == 0 else False

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=baseline_vals_scaled,
            mode='lines',
            name='Linear',
            showlegend=showlegend,
            line=dict(dash='dash', color=color_linear)
        ), row=1, col=i+1)

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=safe_vals,
            mode='lines',
            name='Safe-DDPG',
            showlegend=showlegend,
            line=dict(color=color_safe)
        ), row=1, col=i+1)

    # 保证仅在第一个子图显示y轴标题，第三个子图显示x轴标题
    fig.update_yaxes(title_text="Q (MVar)", row=1, col=1)
    fig.update_xaxes(title=dict(text="Voltage (kV)", standoff=25), row=1, col=3)
    fig.update_layout(
        width=1400,
        height=500,
        showlegend=True,
        xaxis=dict(
            showline=True,
            mirror=True,
            showgrid=True,
        ),
        yaxis=dict(
            showline=True,
            mirror=True,
            showgrid=True,
        ),
    )

    fig.show()


def x_policy_plotly(policy_net):
    """
    用 Plotly 绘制不同拓扑下的 RLC-FT 策略曲线，所有情形绘制在单个图中
    """
    import plotly.graph_objects as go
    fig = go.Figure()
    N = 400
    for i in range(5):
        policy_vals = []
        # 对于每个拓扑情形，通过 reset_topo 获得对应拓扑设定
        state, topo, senario = env.reset_topo(seed=i)
        topo_tensor = torch.cuda.FloatTensor(topo).unsqueeze(0)
        for j in range(N):
            state_tensor = torch.tensor([[0.80 + 0.001 * j]])
            action_tensor = policy_net[2](state_tensor, topo_tensor)
            policy_vals.append(float(-action_tensor.detach().cpu().numpy()[0]))
        policy_vals_smoothed = moving_average(np.array(policy_vals), n=20)
        x_vals = np.linspace(10, 14, N)
        fig.add_trace(go.Scatter(x=x_vals, y=policy_vals_smoothed,
                                 mode='lines',
                                 name=f'Topology {i}'))
    fig.update_layout(
        font=dict(size=16),
        width=700,
        height=500,
        # margin=dict(l=30, r=30, t=30, b=30),   # Uncomment to adjust margins
        margin=dict(r=30,t=30,b=60),
        xaxis_title='Voltage (kV)',
        yaxis_title='Q (MVar)',
        xaxis=dict(
            showgrid=True,
            tickfont=dict(size=12),
        ),
        yaxis=dict(
            tickfont=dict(size=12),
            showgrid=True,
            zeroline=True,
            zerolinecolor='lightgray'
        ),
        legend=dict(
            x=1,
            y=1,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255,255,255,1.0)'
        ),
    )

    fig.show()

### Load model

In [4]:
agent_num = 5
agent_policy_net = []
safe_agent_net = []

### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=256)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=2048).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    #value_net_dict = torch.load(f'check_points/value_net/2023-06-19/Step_200_Seed_12_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-07-05/Step_300_Seed_45_a{i}.pth')
    # policy_net_dict = torch.load(f'check_points/policy_net/2023-08-15/Step_900_Seed_33_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-09-21/Step_900_Seed_10_a{i}.pth')
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_600_Seed_12_a{i}.pth'))
    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(agent_num):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/stable_ddpg/policy_net_checkpoint_a{i}.pth')
    safe_agent_net[i].load_state_dict(policy_net_dict)

In [8]:
policy_plotly(agent_policy_net, topology)
# safe_net_plotly(safe_agent_net)

### Flexible NN Contoller

In [7]:
### test our controller
voltage = []
q = []
cost = []
success_list = []
fail_list = []
entire_list = []
control_cost = []
reward_list = []
object_cost = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost = []
    abnormal_stop = False

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 1.0* action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list.append((episode,step))
            logger.success('episode {} stable at {} steps',success_list[-1][0], success_list[-1][1])
            break

        voltage.append(state)

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list.append(episode_reward)
    control_cost.append(episode_control)
    object_cost.append(np.sum(cost))

    if (not done) and (abnormal_stop == False):
        entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list))
logger.info('total fail episode is {}', len(fail_list))
logger.info('number of finished at entire episode is {}', len(entire_list))

2025-02-27 10:28:33.979 | SUCCESS  | __main__:<module>:48 - episode 0 stable at 24 steps
2025-02-27 10:28:34.396 | SUCCESS  | __main__:<module>:48 - episode 1 stable at 10 steps
2025-02-27 10:28:35.113 | SUCCESS  | __main__:<module>:48 - episode 2 stable at 21 steps
2025-02-27 10:28:36.225 | SUCCESS  | __main__:<module>:48 - episode 3 stable at 32 steps
2025-02-27 10:28:36.691 | SUCCESS  | __main__:<module>:48 - episode 4 stable at 12 steps
2025-02-27 10:28:37.941 | SUCCESS  | __main__:<module>:48 - episode 5 stable at 36 steps
2025-02-27 10:28:39.058 | SUCCESS  | __main__:<module>:48 - episode 6 stable at 31 steps
2025-02-27 10:28:40.090 | SUCCESS  | __main__:<module>:48 - episode 7 stable at 31 steps
2025-02-27 10:28:44.123 | SUCCESS  | __main__:<module>:48 - episode 8 stable at 124 steps
2025-02-27 10:28:46.923 | SUCCESS  | __main__:<module>:48 - episode 9 stable at 86 steps
2025-02-27 10:28:47.506 | SUCCESS  | __main__:<module>:48 - episode 10 stable at 17 steps
2025-02-27 10:28:48

In [8]:
success_list = np.array(success_list)
print('average recovery step is:')
print(np.mean(success_list[:,1]))
print(np.std(success_list[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost))
print(np.std(control_cost))
print('the total cost is:')
print(object_cost)
print(np.mean(object_cost))
print(np.std(object_cost))


average recovery step is:
38.64
32.32553789188975
average reactive power cost is:
140.5893424768255
195.849190257599
the total cost is:
[437.60929479951005, 119.96071816246987, 373.19832072109597, 443.7058825898088, 210.62947167450417, 757.4527250181194, 392.3003163697716, 735.0099603524801, 2576.7146575623633, 1033.0798119913516, 328.400263142729, 704.5233523341334, 266.8768194886957, 1011.5162893543418, 147.51530618604156, 313.9669351384316, 914.8257680918871, 609.4902203875502, 756.209539750806, 384.77093614435194, 0.0, 130.16157436583333, 419.20803105534236, 207.57343863857054, 901.8450402305428, 207.50193993834716, 139.30343919160947, 1020.0307627454736, 409.2986561986554, 82.4719055318557, 268.8522903234471, 734.5130483587657, 1811.1587100355518, 788.5279549664989, 1885.2484961122552, 905.3186368058534, 988.1096140103946, 793.9643579310134, 26.91069180539673, 1761.1768607636134, 273.97104872560186, 841.7491087227094, 210.0734236984085, 441.46471992075305, 352.0577634306893, 1111.

In [10]:
### test our controller without topology change
voltage_ = []
q_ = []
cost_ = []
success_list_ = []
fail_list_ = []
entire_list_ = []
control_cost_ = []
reward_list_ = []
object_cost_list_ = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = 1/env.topology_init
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost_ = []
    abnormal_stop = False

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 0.8* action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list_.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage_ violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list_.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list_.append((episode,step))
            logger.success('stable at {}',success_list_[-1])
            break

        voltage_.append(state)

        q_.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost_.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list_.append(episode_reward)
    control_cost_.append(episode_control)
    object_cost_list_.append(np.sum(cost_))

    if (not done) and (abnormal_stop == False):
        entire_list_.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list_))
logger.info('total fail episode is {}', len(fail_list_))
logger.info('number of finished at entire episode is {}', len(entire_list_))

2025-02-26 17:11:43.012 | SUCCESS  | __main__:<module>:49 - stable at (0, 5)
2025-02-26 17:11:43.115 | SUCCESS  | __main__:<module>:49 - stable at (1, 2)
2025-02-26 17:11:43.312 | SUCCESS  | __main__:<module>:49 - stable at (2, 7)
2025-02-26 17:11:43.440 | SUCCESS  | __main__:<module>:49 - stable at (3, 4)
2025-02-26 17:11:43.592 | SUCCESS  | __main__:<module>:49 - stable at (4, 5)
2025-02-26 17:11:43.835 | SUCCESS  | __main__:<module>:49 - stable at (5, 9)
2025-02-26 17:11:43.890 | SUCCESS  | __main__:<module>:49 - stable at (6, 1)
2025-02-26 17:11:44.550 | SUCCESS  | __main__:<module>:49 - stable at (7, 27)
2025-02-26 17:11:44.961 | SUCCESS  | __main__:<module>:49 - stable at (8, 16)
2025-02-26 17:11:45.251 | SUCCESS  | __main__:<module>:49 - stable at (9, 11)
2025-02-26 17:11:45.541 | SUCCESS  | __main__:<module>:49 - stable at (10, 10)
2025-02-26 17:11:45.715 | SUCCESS  | __main__:<module>:49 - stable at (11, 6)
2025-02-26 17:11:45.858 | SUCCESS  | __main__:<module>:49 - stable at 

In [11]:
success_list_ = np.array(success_list_)
print('average recovery step is:')
print(np.mean(success_list_[:,1]))
print(np.std(success_list_[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost_))
print(np.std(control_cost_))
print('the total cost is:')
print(np.mean(object_cost_list_))
print(np.std(object_cost_list_))

average recovery step is:
6.665
7.289223209643124
average reactive power cost is:
27.94394344393491
54.532998712140966
the total cost is:
115.45291077537155
136.36839270568598


### baseline

In [10]:
### test the base line controller
num_agent = 5
voltage = []
q = []
base_cost = []
base_succ_list = []
base_fail_list = []
base_entire_list = []
base_control_cost = []
base_reward_list = []
base_object_cost = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    base_cost = []
    abnormal_stop = False

    for step in range(step_num):
        state1 = np.asarray(state-env.vmax + 0.001)
        state2 = np.asarray(env.vmin-state + 0.001)
        d_v = (np.maximum(state1, 0)-np.maximum(state2, 0)).reshape((num_agent,1))
        
        action = (last_action - 10*d_v)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            base_succ_list.append((episode,step))
            logger.success('stable at {}',base_succ_list[-1])
            break

        voltage.append(state)

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        base_cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    base_control_cost.append(episode_control)
    base_reward_list.append(episode_reward)
    base_object_cost.append(np.sum(base_cost))
    
    if (not done) and (abnormal_stop == False):
        base_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(base_succ_list))
logger.info('total fail episode is {}', len(base_fail_list))
logger.info('number of finished at entire episode is {}', len(base_entire_list))

2025-03-19 17:01:31.851 | SUCCESS  | __main__:<module>:47 - stable at (0, 25)
2025-03-19 17:01:31.972 | SUCCESS  | __main__:<module>:47 - stable at (1, 7)
2025-03-19 17:01:32.412 | SUCCESS  | __main__:<module>:47 - stable at (2, 38)
2025-03-19 17:01:32.662 | SUCCESS  | __main__:<module>:47 - stable at (3, 22)
2025-03-19 17:01:32.902 | SUCCESS  | __main__:<module>:47 - stable at (4, 20)
2025-03-19 17:01:33.409 | SUCCESS  | __main__:<module>:47 - stable at (5, 46)
2025-03-19 17:01:33.466 | SUCCESS  | __main__:<module>:47 - stable at (6, 2)
2025-03-19 17:01:34.115 | SUCCESS  | __main__:<module>:47 - stable at (7, 59)
2025-03-19 17:01:34.829 | SUCCESS  | __main__:<module>:47 - stable at (8, 66)
2025-03-19 17:01:34.944 | SUCCESS  | __main__:<module>:47 - stable at (9, 10)
2025-03-19 17:01:35.428 | SUCCESS  | __main__:<module>:47 - stable at (10, 41)
2025-03-19 17:01:35.653 | SUCCESS  | __main__:<module>:47 - stable at (11, 20)
2025-03-19 17:01:35.811 | SUCCESS  | __main__:<module>:47 - stab

In [11]:
base_succ_list = np.array(base_succ_list)
print('average recovery step is:')
print(np.mean(base_succ_list[:,1]))
print(np.std(base_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(base_control_cost))
print(np.std(base_control_cost))
print('the total cost is:')
print(base_object_cost)
print(np.mean(base_object_cost))
print(np.std(base_object_cost))


average recovery step is:
20.995
18.244861605394544
average reactive power cost is:
93.24602557422712
167.73421448165357
the total cost is:
[331.4381230096074, 79.37194620263368, 539.1592184469716, 284.34496406540705, 222.78735969737323, 754.1682702813467, 29.84013426138405, 1136.411871022985, 1056.9603629466496, 122.90933453666014, 560.9620343802666, 309.9329477329244, 162.46817297672234, 345.22781590281625, 84.00087648036495, 77.87157459061811, 447.9030283239126, 1143.0472313473442, 176.88333267800874, 557.5911198092685, 0.0, 91.96842769393515, 86.63425232033393, 337.90413282584814, 167.46888434182415, 81.27101095991333, 313.7137401372946, 213.78912549363267, 197.31651844449183, 233.30639714747372, 11.678091367223365, 208.69666857425483, 774.4340601730597, 310.12122161707396, 699.6066092539205, 165.95381847850743, 1365.3797876968981, 1149.4435999178938, 20.163409750434248, 889.4776413105868, 263.332872379289, 75.72796956047941, 242.8263436597923, 38.36772898430843, 253.7053771614978,

### Safe DDPG

In [7]:
### test the safe policy net
num_agent = 5
safe_voltage = []
safe_q = []
safe_cost = []
safe_succ_list = []
safe_fail_list = []
safe_entire_list = []
safe_contorl_cost = []
safe_reward_list = []
safe_object_cost = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    safe_cost = []
    abnormal_stop = False

    for step in range(step_num):
        action = []
        for i in range(num_agent):
            action_agent = safe_agent_net[i].get_action(torch.cuda.FloatTensor([state[i]]).float().reshape(1,1))
            action.append(action_agent)
        
        action = last_action - 5*np.asarray(action).reshape((num_agent, 1))
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            safe_succ_list.append((episode,step))
            logger.success('stable at {}',safe_succ_list[-1])
            break
        safe_voltage.append(state)

        safe_q.append(action)

        state = next_state
        
        episode_reward += reward
        
        safe_cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    safe_contorl_cost.append(episode_control)
    safe_reward_list.append(episode_reward)
    safe_object_cost.append(np.sum(safe_cost))

    if (not done) and (abnormal_stop == False):
        safe_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(safe_succ_list))
logger.info('total fail episode is {}', len(safe_fail_list))
logger.info('number of finished at entire episode is {}', len(safe_entire_list))


2025-02-26 16:32:51.789 | SUCCESS  | __main__:<module>:48 - stable at (0, 32)
2025-02-26 16:32:51.944 | SUCCESS  | __main__:<module>:48 - stable at (1, 5)
2025-02-26 16:32:53.087 | SUCCESS  | __main__:<module>:48 - stable at (2, 53)
2025-02-26 16:32:53.631 | SUCCESS  | __main__:<module>:48 - stable at (3, 23)
2025-02-26 16:32:54.197 | SUCCESS  | __main__:<module>:48 - stable at (4, 21)
2025-02-26 16:32:55.642 | SUCCESS  | __main__:<module>:48 - stable at (5, 66)
2025-02-26 16:32:55.691 | SUCCESS  | __main__:<module>:48 - stable at (6, 0)
2025-02-26 16:32:57.145 | SUCCESS  | __main__:<module>:48 - stable at (7, 63)
2025-02-26 16:32:59.082 | SUCCESS  | __main__:<module>:48 - stable at (8, 83)
2025-02-26 16:32:59.340 | SUCCESS  | __main__:<module>:48 - stable at (9, 10)
2025-02-26 16:33:00.446 | SUCCESS  | __main__:<module>:48 - stable at (10, 51)
2025-02-26 16:33:00.898 | SUCCESS  | __main__:<module>:48 - stable at (11, 18)
2025-02-26 16:33:01.171 | SUCCESS  | __main__:<module>:48 - stab

In [8]:
safe_succ_list = np.array(safe_succ_list)
print('average recovery step is:')
print(np.mean(safe_succ_list[:,1]))
print(np.std(safe_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(safe_contorl_cost))
print(np.std(safe_contorl_cost))
print('the total cost is:')
print(safe_object_cost)
print(np.mean(safe_object_cost))
print(np.std(safe_object_cost))

average recovery step is:
23.745
22.76510432657843
average reactive power cost is:
113.28480666367739
201.52561556901605
the total cost is:
[403.27795744816035, 55.325414340995955, 727.1358452981689, 294.4688507194921, 202.61152792137142, 1044.1712508603164, 0.0, 1231.4248697584305, 1315.2206187810293, 118.93681244404073, 670.783963796241, 274.1472899338466, 127.5456695089633, 423.3392394645283, 58.6587445928317, 0.0, 580.6666386539297, 1253.1081617403827, 184.43242772020008, 585.2495710662595, 0.0, 63.72166418513142, 64.9542762283673, 316.4960719270453, 150.98023253070193, 65.17465192296146, 332.1882578868361, 160.70081383701287, 194.89533951250965, 247.45159738431687, 0.0, 243.2038569518022, 1033.2732244977092, 363.1982795740444, 828.2557008552561, 132.2999422816827, 1487.2568119668974, 1252.4900974806337, 20.04220935187984, 1226.2190909731132, 260.4268165919477, 0.0, 186.11924126336132, 0.0, 353.5256759358745, 544.4987121291903, 937.807542688232, 28.16757670679894, 424.8058125176003